# KOSIS early BGE retrieval-assisted extraction

`is_claim=True` 원문을 BGE-M3로 먼저 검색하고, reranker Top-5와 KOSIS ITEM/단위 메타를 HCX 구조화의 참고 정보로 전달합니다. 기사에 없는 값이나 기간은 후보에서 복사하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/rnwjdgus03/NLP_05-Team-Project-3.git'
BRANCH = 'codex/repro-baseline-20260727'
REPO_DIR = Path('/content/NLP_05-Team-Project-3')
DRIVE_ROOT = Path('/content/drive/MyDrive/NLP_05-Team-Project-3')
INPUT_DIR = DRIVE_ROOT / 'inputs'
INDEX_DIR = DRIVE_ROOT / 'indexes' / 'kosis_bge_m3'
RUN_DIR = DRIVE_ROOT / 'runs' / 'early_bge_rag'

INPUT_CSV = INPUT_DIR / 'is_claim_news_4000_true.csv'
CANDIDATES_CSV = RUN_DIR / 'early_bge_candidates_top20.csv'
CONTEXT_CSV = RUN_DIR / 'early_bge_context_top5.csv'
UNIQUE_TABLES_CSV = RUN_DIR / 'early_bge_unique_top5_tables.csv'
META_CSV = RUN_DIR / 'early_bge_meta_index.csv'
HCX_OUTPUT_CSV = RUN_DIR / 'hcx_early_bge_extracted.csv'
READY_CSV = RUN_DIR / 'hcx_early_bge_kosis_ready.csv'
REJECTED_CSV = RUN_DIR / 'hcx_early_bge_kosis_rejected.csv'
HCX_LIMIT = 20

INPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('input:', INPUT_CSV)
print('index:', INDEX_DIR)
print('run:', RUN_DIR)

In [ ]:
import os
import subprocess
import sys

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
os.chdir(REPO_DIR)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'numpy>=1.26,<3', 'sentence-transformers>=3.4,<6',
    'transformers>=4.45,<6', 'requests>=2.31,<3', 'python-dotenv>=1.0,<2'
], check=True)
print('repo:', REPO_DIR)

In [ ]:
from google.colab import files

if not INPUT_CSV.exists():
    print('is_claim_news_4000_true.csv 파일을 선택하세요.')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError('CSV 파일 하나만 업로드하세요.')
    INPUT_CSV.write_bytes(next(iter(uploaded.values())))

required_index_files = [INDEX_DIR / 'manifest.json', INDEX_DIR / 'tables.csv', INDEX_DIR / 'embeddings.npy']
missing = [str(path) for path in required_index_files if not path.exists()]
if missing:
    raise FileNotFoundError('BGE 인덱스 파일이 없습니다: ' + ', '.join(missing))
print('input and index: ready')

In [ ]:
command = [
    sys.executable, str(REPO_DIR / 'kosis_early_retrieve.py'),
    '--input', str(INPUT_CSV),
    '--output-candidates', str(CANDIDATES_CSV),
    '--output-context', str(CONTEXT_CSV),
    '--semantic-index', str(INDEX_DIR),
    '--semantic-top-k', '20',
    '--rerank-top-k', '20',
    '--context-top-k', '5',
    '--checkpoint-every', '10',
    '--device', 'cuda',
]
subprocess.run(command, check=True)

In [ ]:
import pandas as pd

candidates = pd.read_csv(CANDIDATES_CSV, encoding='utf-8-sig')
contexts = pd.read_csv(CONTEXT_CSV, encoding='utf-8-sig')
print('claims:', contexts['claim_id'].nunique())
print('candidate rows:', len(candidates))
display(candidates.head(20))

top5 = candidates[candidates['candidate_rank'] <= 5].copy()
unique_tables = top5.drop_duplicates(['org_id', 'tbl_id'])[
    ['org_id', 'tbl_id', 'tbl_name', 'category_path']
]
unique_tables.to_csv(UNIQUE_TABLES_CSV, index=False, encoding='utf-8-sig')
print('unique Top-5 tables:', len(unique_tables))

## KOSIS 메타 보강

Colab 보안 비밀에 `KOSIS_API_KEY`를 등록한 뒤 실행합니다. 고유 Top-5 통계표만 조회하며, 중단되면 같은 셀을 다시 실행해 이어받습니다.

In [ ]:
from google.colab import userdata

if not os.environ.get('KOSIS_API_KEY'):
    os.environ['KOSIS_API_KEY'] = userdata.get('KOSIS_API_KEY') or ''
if not os.environ.get('KOSIS_API_KEY'):
    raise RuntimeError('Colab 보안 비밀에 KOSIS_API_KEY를 등록하세요.')

meta_command = [
    sys.executable, str(REPO_DIR / 'kosis_build_meta_index.py'),
    '--table-index', str(UNIQUE_TABLES_CSV),
    '--out', str(META_CSV),
    '--delay', '0.10',
    '--resume',
]
subprocess.run(meta_command, check=True)

In [ ]:
enrich_command = [
    sys.executable, str(REPO_DIR / 'kosis_early_retrieve.py'),
    '--reuse-candidates', str(CANDIDATES_CSV),
    '--output-candidates', str(CANDIDATES_CSV),
    '--output-context', str(CONTEXT_CSV),
    '--meta-index', str(META_CSV),
    '--context-top-k', '5',
]
subprocess.run(enrich_command, check=True)

contexts = pd.read_csv(CONTEXT_CSV, encoding='utf-8-sig')
display(contexts.head())

## HCX 파일럿과 게이트

기본은 20건만 실행합니다. 결과를 확인한 뒤 위 설정 셀의 `HCX_LIMIT`을 `0`으로 바꾸면 남은 claim을 이어받아 처리합니다. baseline 출력과 경로를 분리합니다.

In [ ]:
from google.colab import userdata

if not os.environ.get('CLOVA_API_KEY'):
    os.environ['CLOVA_API_KEY'] = userdata.get('CLOVA_API_KEY') or ''
if not os.environ.get('CLOVA_API_KEY'):
    raise RuntimeError('Colab 보안 비밀에 CLOVA_API_KEY를 등록하세요.')

hcx_command = [
    sys.executable, str(REPO_DIR / 'extract_hcx.py'),
    '--input', str(INPUT_CSV),
    '--retrieval-context', str(CONTEXT_CSV),
    '--output', str(HCX_OUTPUT_CSV),
    '--model', 'HCX-007',
    '--limit', str(HCX_LIMIT),
    '--sleep', '0.5',
]
subprocess.run(hcx_command, check=True)

In [ ]:
gate_command = [
    sys.executable, str(REPO_DIR / 'prepare_kosis_mapping_input.py'),
    '--input', str(HCX_OUTPUT_CSV),
    '--output', str(READY_CSV),
    '--rejected-output', str(REJECTED_CSV),
]
subprocess.run(gate_command, check=True)

ready = pd.read_csv(READY_CSV, encoding='utf-8-sig')
rejected = pd.read_csv(REJECTED_CSV, encoding='utf-8-sig')
print('READY:', len(ready))
print('rejected:', len(rejected))
if 'mapping_exclusion_code' in rejected.columns:
    display(rejected['mapping_exclusion_code'].value_counts(dropna=False))